In [ ]:
import pandas as pd
import glob

# 1. Get a list of all Cyclistic CSV files in your folder
# (Make sure all 12 monthly CSVs are saved in the same folder as this script)
csv_files = glob.glob('*.csv')

# 2. Read and combine all CSV files into a single DataFrame
print("Combining files... This might take a moment.")
df_list = [pd.read_csv(file) for file in csv_files]
df = pd.concat(df_list, ignore_index=True)

print(f"Combined successfully! Total raw rows: {df.shape[0]:,}")

In [2]:
df.to_csv('master_cyclistic_data.csv', index=False)
print("Master CSV saved successfully!")

Master CSV saved successfully!


In [7]:
df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'member_casual', 'ride_length_mins', 'day_of_week'],
      dtype='str')

In [8]:
# overall average ride length
df['ride_length_mins'].mean()

np.float64(15.126809214620769)

In [10]:
# average of ride length for casual and members
df.groupby('member_casual')['ride_length_mins'].mean()

member_casual
casual    20.302232
member    12.402564
Name: ride_length_mins, dtype: float64

In [13]:
df['ride_length_mins'].min()

np.float64(1.00005)

In [15]:
df.groupby('day_of_week')['ride_length_mins'].count()

day_of_week
Friday       606709
Monday       547395
Saturday     617779
Sunday       501241
Thursday     588624
Tuesday      595320
Wednesday    592377
Name: ride_length_mins, dtype: int64

In [21]:
df['day_of_week'].mode()

0    Saturday
Name: day_of_week, dtype: str

In [24]:
# Average ride_length for users by day_of_week

df.groupby(["member_casual",'day_of_week'])['ride_length_mins'].mean()

member_casual  day_of_week
casual         Friday         19.984661
               Monday         20.321269
               Saturday       23.074811
               Sunday         23.606787
               Thursday       17.709609
               Tuesday        17.444269
               Wednesday      16.605098
member         Friday         12.382172
               Monday         12.087359
               Saturday       13.820850
               Sunday         13.686308
               Thursday       11.884472
               Tuesday        11.919556
               Wednesday      11.807273
Name: ride_length_mins, dtype: float64

In [25]:
# Number of rides for users by day_of_week by counting tripID

df.groupby('day_of_week')['ride_id'].count()

day_of_week
Friday       606709
Monday       547395
Saturday     617779
Sunday       501241
Thursday     588624
Tuesday      595320
Wednesday    592377
Name: ride_id, dtype: int64

In [28]:
# adding month column - It will run thru' Oct'25 to Sep'26 - but stored only as Oct, Nov, Dec, Jan....Sep
df['started_at'] = pd.to_datetime(df['started_at'])
df['month'] = df['started_at'].dt.month_name()

In [30]:
df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,member_casual,ride_length_mins,day_of_week,month
0,A835B6FFB85FF2F8,electric_bike,2026-07-15 20:19:48.831,2026-07-15 21:22:00.281,Burnham Greenway & 105th St,CHI00863,Burnham Greenway & 105th St,CHI00863,casual,62.190833,Wednesday,July
1,880813FDABFBAEF9,classic_bike,2026-07-10 23:35:22.204,2026-07-10 23:44:25.580,California Ave & Milwaukee Ave,CHI00307,California Ave & Division St,CHI00340,casual,9.056267,Friday,July
2,12AB6CD18F6C311D,classic_bike,2026-07-25 00:22:16.313,2026-07-25 00:23:43.303,California Ave & Milwaukee Ave,CHI00307,California Ave & Milwaukee Ave,CHI00307,casual,1.449833,Saturday,July
3,B79A9800B7B4DDA9,classic_bike,2026-07-04 11:18:15.715,2026-07-04 11:22:56.951,Granville Ave & Clark St,CHI02156,Granville Ave & Clark St,CHI02156,casual,4.687267,Saturday,July
4,99AD35C0916D16A8,classic_bike,2026-07-15 18:36:05.398,2026-07-15 19:49:07.601,Burnham Greenway & 105th St,CHI00863,Burnham Greenway & 105th St,CHI00863,casual,73.036717,Wednesday,July


In [31]:
df.groupby('month')['ride_length_mins'].sum()

month
April        4.143616e+06
August       8.767480e+06
December     1.193820e+06
February     1.728572e+06
January      1.207369e+06
July         9.058704e+06
June         8.233192e+06
March        2.961166e+06
May          6.962843e+06
November     3.142304e+06
October      6.335891e+06
September    7.520224e+06
Name: ride_length_mins, dtype: float64

In [36]:
count_month_type = df.groupby(['month','member_casual'])[['ride_length_mins']].count()

In [41]:
# 1. Pivot the 'month' index level into columns
pivot_df = count_month_type.unstack(level='month')

# 2. Define your custom order starting from October and ending in September
oct_to_sep_order = [
    'October', 'November', 'December', 'January', 'February', 'March', 
    'April', 'May', 'June', 'July', 'August', 'September'
]

# 3. Reindex the columns using your custom order
# (We use axis=1 and pass the order to the second level of the MultiIndex columns)
final_df = pivot_df.reindex(columns=oct_to_sep_order, level='month')

# Display the clean result
final_df

ride_length_mins                                             \
month                  October November December January February   March   
member_casual                                                               
casual                  142180    63617    18640   16395    29369   59799   
member                  271971   169366    76492   79736   116797  159242   

                                                                 
month           April     May    June    July  August September  
member_casual                                                    
casual          86529  156703  196947  224405  216458    185432  
member         212525  273876  306437  334007  337443    315079

In [43]:
# Mean ride length in minutes by month

mean_ride_length = df.groupby(['month','member_casual'])[['ride_length_mins']].mean()

pivot_mean = mean_ride_length.unstack(level='month')

final_mean = pivot_mean.reindex(columns=oct_to_sep_order, level='month')

final_mean

ride_length_mins                                              \
month                  October   November   December    January   February   
member_casual                                                                
casual               20.630885  17.071643  14.401741  14.708908  15.535942   
member               12.510862  12.140911  12.097629  12.117691  10.893234   

                                                                     \
month             March      April        May       June       July   
member_casual                                                         
casual         18.82395  18.799173  22.225342  21.648563  21.399937   
member         11.52656  11.843044  12.706717  12.953959  12.743600   

                                     
month             August  September  
member_casual                        
casual         20.495881  19.569669  
member         12.834710  12.350493

In [44]:
# Max ride length in minutes by month
max_ride_length = df.groupby(['month','member_casual'])[['ride_length_mins']].max()

pivot_max = max_ride_length.unstack(level='month')

final_max = pivot_max.reindex(columns=oct_to_sep_order, level='month')

final_max

ride_length_mins                                         \
month                  October     November     December      January   
member_casual                                                           
casual             1439.975950  1439.274933  1431.303517  1439.072633   
member             1439.827217  1434.759350  1418.238100  1414.889917   

                                                                   \
month             February        March        April          May   
member_casual                                                       
casual         1415.035467  1431.961833  1428.452600  1438.687167   
member         1424.639833  1405.116700  1358.445183  1362.277117   

                                                                   
month                 June         July       August    September  
member_casual                                                      
casual         1433.939933  1431.184633  1438.625367  1428.976817  
member         1345.949683  1420.070283  1401.831450  1435.603217

In [45]:
# Min ride length in minutes by month
min_ride_length = df.groupby(['month','member_casual'])[['ride_length_mins']].min()

pivot_min = min_ride_length.unstack(level='month')

final_min = pivot_min.reindex(columns=oct_to_sep_order, level='month')

final_min

ride_length_mins                                          \
month                  October  November  December   January  February   
member_casual                                                            
casual                1.001217  1.007967  1.001083  1.002467  1.014183   
member                1.001383  1.000433  1.001050  1.000117  1.000183   

                                                                           \
month             March     April       May      June      July    August   
member_casual                                                               
casual         1.000417  1.002783  1.000300  1.001533  1.000400  1.000467   
member         1.000050  1.000233  1.000317  1.001700  1.000817  1.000667   

                         
month         September  
member_casual            
casual         1.001833  
member         1.000117

In [48]:
# Mode of the day of the week
# Use .agg() to apply the mode function to each group
mode_day = df.groupby(['month', 'member_casual'])['day_of_week'].agg(lambda x: x.mode().iloc[0]).reset_index()

# Convert the 'month' column to a Categorical type with your custom order
mode_day['month'] = pd.Categorical(mode_day['month'], categories=oct_to_sep_order, ordered=True)

# Sort by month and rider type
mode_day = mode_day.sort_values(by=['month', 'member_casual']).reset_index(drop=True)

# Display the clean result
print(mode_day)

        month member_casual day_of_week
0     October        casual    Saturday
1     October        member    Thursday
2    November        casual    Saturday
3    November        member   Wednesday
4    December        casual    Saturday
5    December        member     Tuesday
6     January        casual     Tuesday
7     January        member     Tuesday
8    February        casual      Friday
9    February        member   Wednesday
10      March        casual    Saturday
11      March        member      Monday
12      April        casual    Thursday
13      April        member    Thursday
14        May        casual    Saturday
15        May        member      Friday
16       June        casual    Saturday
17       June        member     Tuesday
18       July        casual    Saturday
19       July        member   Wednesday
20     August        casual    Saturday
21     August        member      Monday
22  September        casual    Saturday
23  September        member     Tuesday


In [49]:
# 1. Pivot the dataframe (Rows = member_casual, Columns = month, Values = day_of_week)
pivot_mode = mode_day.pivot(index='member_casual', columns='month', values='day_of_week')

# 2. Define your custom order from October to September
oct_to_sep_order = [
    'October', 'November', 'December', 'January', 'February', 'March', 
    'April', 'May', 'June', 'July', 'August', 'September'
]

# 3. Reindex the columns to force the custom chronological order
final_mode_df = pivot_mode.reindex(columns=oct_to_sep_order)

# Display the final table
final_mode_df

month,October,November,December,January,February,March,April,May,June,July,August,September
member_casual,,,,,,,,,,,,
casual,Saturday,Saturday,Saturday,Tuesday,Friday,Saturday,Thursday,Saturday,Saturday,Saturday,Saturday,Saturday
member,Thursday,Wednesday,Tuesday,Tuesday,Wednesday,Monday,Thursday,Friday,Tuesday,Wednesday,Monday,Tuesday


In [54]:
# average by day of the week

# 1. Calculate the mean (assuming you haven't run it yet)
avg_by_day = df.groupby(['month', 'day_of_week'])['ride_length_mins'].mean()

# 2. Pivot the 'month' level into columns
pivot_avg = avg_by_day.unstack(level='month')

# 3. Define the custom order for rows (Days of the week) and columns (Months)
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
months_order = [
    'October', 'November', 'December', 'January', 'February', 'March', 
    'April', 'May', 'June', 'July', 'August', 'September'
]

# 4. Reindex both rows (index) and columns using the custom orders
final_avg_df = pivot_avg.reindex(index=days_order, columns=months_order)

# Display the final beautiful table
final_avg_df

month,October,November,December,January,February,March,April,May,June,July,August,September
day_of_week,,,,,,,,,,,,
Monday,14.579166,12.234137,11.665443,12.873642,12.238375,14.572726,12.463227,17.101606,14.770059,15.219560,14.725737,15.209049
Tuesday,13.102689,11.901179,13.888059,11.527501,10.839564,10.826690,13.134058,13.610592,14.454718,14.554509,14.430613,13.241760
Wednesday,12.642145,12.371166,12.115468,11.730622,11.598740,12.193613,12.217581,13.587144,13.365837,14.658550,13.899370,12.539097
Thursday,13.527675,12.183227,11.542019,12.108788,10.556530,11.803323,13.492427,13.302745,14.950901,14.564895,14.254735,13.889424
Friday,15.469690,13.706840,11.902322,13.909872,12.346188,12.275224,14.453950,15.030644,16.846400,16.326511,15.898713,14.886638
Saturday,18.339220,16.511264,13.480855,14.930648,12.175985,16.836435,15.818803,19.322000,20.365171,19.770377,18.342703,17.604852
Sunday,20.156269,15.201494,13.623687,11.852019,14.406772,15.430680,16.694419,19.510480,19.015084,18.803419,18.280389,18.863175


In [55]:
# Split of average ride length by month by day split betwen members and casual

# 1. Calculate the mean including member_casual
avg_split = df.groupby(['month', 'day_of_week', 'member_casual'])['ride_length_mins'].mean()

# 2. Pivot the 'month' level into columns
pivot_split = avg_split.unstack(level='month')

# 3. Define the custom chronological and weekly orders
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
months_order = [
    'October', 'November', 'December', 'January', 'February', 'March', 
    'April', 'May', 'June', 'July', 'August', 'September'
]

# 4. Reindex the rows (level 0 is day_of_week) and the columns (month)
# We use .reindex(index=...) and pass the days_order to the specific level 'day_of_week'
final_split_df = pivot_split.reindex(index=days_order, level='day_of_week').reindex(columns=months_order)

# Display the final comparative table
final_split_df

month                        October   November   December    January  \
day_of_week member_casual                                               
Monday      casual         20.411366  15.455494  12.670174  14.932254   
            member         12.163868  11.384947  11.481913  12.531389   
Tuesday     casual         17.319261  13.588464  13.840191  11.695529   
            member         11.672626  11.459636  13.897969  11.498465   
Wednesday   casual         15.263209  13.567648  13.949302  11.781635   
            member         11.723141  12.065781  11.730833  11.721795   
Thursday    casual         17.228943  13.808121  14.427584  12.843357   
            member         12.012316  11.707056  10.899009  11.969110   
Friday      casual         20.682088  17.129485  12.753855  16.962259   
            member         12.549995  12.299833  11.671026  13.210497   
Saturday    casual         23.115813  20.683151  17.617097  20.677591   
            member         13.886908  13.685211  11.747697  13.255119   
Sunday      casual         26.133293  19.711753  15.036061  15.134793   
            member         14.788255  12.680488  13.169893  10.784697   

month                       February      March      April        May  \
day_of_week member_casual                                               
Monday      casual         18.189259  21.335438  16.499691  24.154348   
            member         10.828311  12.140082  11.232555  13.070953   
Tuesday     casual         11.805993  12.916700  17.859879  18.430309   
            member         10.668590  10.369483  11.594883  11.745597   
Wednesday   casual         14.031217  16.256864  15.099584  17.739218   
            member         11.098244  11.159963  11.354510  12.030701   
Thursday    casual         11.931614  14.910322  17.820658  17.678906   
            member         10.282594  10.952445  11.935424  11.625442   
Friday      casual         16.510789  16.470884  19.382619  20.337760   
            member         11.098166  10.750955  12.111679  12.242102   
Saturday    casual         15.662865  21.848739  20.991167  24.646887   
            member         10.894889  13.304499  12.485585  14.319266   
Sunday      casual         20.159793  20.928541  22.635483  25.805890   
            member         11.946135  12.317895  12.831784  14.131285   

month                           June       July     August  September  
day_of_week member_casual                                              
Monday      casual         20.318348  21.245398  19.810517  20.480025  
            member         12.177428  12.177911  12.226360  12.294762  
Tuesday     casual         18.718508  18.924626  18.926529  16.535798  
            member         12.468167  12.455690  12.334751  11.930042  
Wednesday   casual         17.039578  18.975535  17.524238  15.255460  
            member         11.828999  12.383539  12.213244  11.423065  
Thursday    casual         19.433607  19.091435  18.210352  18.001499  
            member         12.722448  12.110975  12.240705  11.937457  
Friday      casual         21.677067  21.310058  20.365375  19.049521  
            member         13.259967  12.707836  12.748533  12.230363  
Saturday    casual         25.206133  24.516099  22.413316  21.456630  
            member         14.790772  14.244448  14.167957  13.798350  
Sunday      casual         24.028549  23.661371  22.838130  24.002930  
            member         14.134287  13.912178  14.199063  13.955321

In [56]:
# Calculate the number of rides for users by day of the week - by counting ride_id

# 1. Calculate the total number of rides using .count() on ride_id
ride_counts = df.groupby(['month', 'day_of_week', 'member_casual'])['ride_id'].count()

# 2. Pivot the 'month' level into columns
pivot_counts = ride_counts.unstack(level='month')

# 3. Define the custom chronological and weekly orders
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
months_order = [
    'October', 'November', 'December', 'January', 'February', 'March', 
    'April', 'May', 'June', 'July', 'August', 'September'
]

# 4. Reindex the rows (day_of_week) and columns (month) using your custom orders
final_counts_df = pivot_counts.reindex(index=days_order, level='day_of_week').reindex(columns=months_order)

# Display the final ride volume table
final_counts_df

month                      October  November  December  January  February  \
day_of_week member_casual                                                   
Monday      casual           14553      6318      2234     1882      3913   
            member           35141     23967     12230    11320     16514   
Tuesday     casual           11850      6862      3251     2768      3548   
            member           34940     26222     15703    16018     20055   
Wednesday   casual           17177      7068      3048     2446      4447   
            member           48990     27692     14532    14137     21613   
Thursday    casual           20766      7444      2459     2548      4003   
            member           50721     25402     11035    13400     20094   
Friday      casual           24848     10359      2817     2571      5514   
            member           44360     25199     10371    11221     18400   
Saturday    casual           29011     15520      3327     2212      4033   
            member           31124     22911      7940     7587     10977   
Sunday      casual           23975     10046      1504     1968      3911   
            member           26695     17973      4681     6053      9144   

month                      March  April    May   June   July  August  \
day_of_week member_casual                                              
Monday      casual         10881   8482  20138  24296  21015   27107   
            member         30249  27820  35237  51994  41633   55147   
Tuesday     casual          5838  11299  14898  27329  24471   23157   
            member         26687  34692  38501  58657  50953   49675   
Wednesday   casual          5505  11900  15286  16048  33241   23101   
            member         21640  39737  40778  38362  63077   49663   
Thursday    casual          6316  14381  15081  22458  30563   23631   
            member         23063  39977  39347  45176  56377   46412   
Friday      casual          7291  13912  23437  31459  35241   33072   
            member         20069  29274  44605  42373  48533   46893   
Saturday    casual         13826  13243  37891  47434  43877   48148   
            member         19621  20550  40331  41193  37682   46947   
Sunday      casual         10142  13312  29972  27923  35997   38242   
            member         17913  20475  35077  28682  35752   42706   

month                      September  
day_of_week member_casual             
Monday      casual             23258  
            member             42066  
Tuesday     casual             25047  
            member             62899  
Wednesday   casual             21226  
            member             51663  
Thursday    casual             21879  
            member             46091  
Friday      casual             29172  
            member             45718  
Saturday    casual             35983  
            member             36411  
Sunday      casual             28867  
            member             30231

ANALYSIS BY STATION

In [61]:
# Top 10 most popular start_station_name

# 1. Filter and get the top 10 stations for Casual riders
casual_stations = (
    df[df['member_casual'] == 'casual']['start_station_name']
    .value_counts()
    .head(10)
    .reset_index()
)
casual_stations.columns = ['Casual Station Name', 'Casual Ride Count']

# 2. Filter and get the top 10 stations for Member riders
member_stations = (
    df[df['member_casual'] == 'member']['start_station_name']
    .value_counts()
    .head(10)
    .reset_index()
)
member_stations.columns = ['Member Station Name', 'Member Ride Count']

# 3. Combine both lists side by side into a single data frame for easy comparison
top_stations_df = pd.concat([casual_stations, member_stations], axis=1)

# Display the final comparison table
top_stations_df

,Casual Station Name,Casual Ride Count,Member Station Name,Member Ride Count
0,Navy Pier,44897,Canal St & Madison St,19914
1,DuSable Lake Shore Dr & Monroe St,26062,State St & Chicago Ave,18901
2,Michigan Ave & Oak St,18329,Clinton St & Jackson Blvd,17560
3,DuSable Lake Shore Dr & North Blvd,15003,Clinton St & Madison St,16592
4,Millennium Park,14935,Wells St & Concord Ln,15968
5,Shedd Aquarium,14830,Wells St & Elm St,15838
6,Theater on the Lake,13026,Kingsbury St & Kinzie St 2,15727
7,Dusable Harbor,12240,Clinton St & Washington Blvd 2,15166
8,Adler Planetarium,9164,Wells St & Hubbard St,14996
9,Field Museum,9101,Wells St & Huron St,14496


In [62]:
# PEAK HOURS OF THE DAY

# 1. Ensure the date column is in datetime format
df['started_at'] = pd.to_datetime(df['started_at'])

# 2. Extract the hour component (0 to 23) into a new column
df['hour'] = df['started_at'].dt.hour

# 3. Group by hour and rider type, then count the total rides
hourly_counts = df.groupby(['hour', 'member_casual'])['ride_id'].count()

# 4. Pivot the dataframe so hours are rows and rider types are columns
hourly_pivot = hourly_counts.unstack(level='member_casual')

# Display the clean comparative table
hourly_pivot

member_casual,casual,member
hour,,
0,27833,21173
1,17830,12339
2,11050,6813
3,6491,4287
4,5050,5774
5,9421,26614
6,19978,80963
7,35924,162353
8,50469,197617
